In [31]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
import pickle
load_dotenv()
with open("bfl_store.pkl", "rb") as f:
    store = pickle.load(f)

In [21]:
# store['vectors']

In [32]:
# store
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)


In [33]:
GOOD_PROMPT = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- If the answer is not in the CONTEXT, say exactly:
  "I don't have this information in the provided documents."
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")

In [34]:
chunk_vectors = store['vectors']
chunk_texts = store['text']
chunk_meta = store['meta']
model_name = store['model']

In [25]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(model_name)
def retrieve_similar_chunks(query, top_k=3):
     # Convert the query into an embedding
    query_vector = model.encode([query])

    # Compare query with all chunk vectors using cosine similarity
    scores = cosine_similarity(query_vector, chunk_vectors)[0]

    # filter based on thresold
    # scores

    # Get indexes of top matching chunks
    top_indexes = scores.argsort()[::-1][:top_k]

    # Return (score, index)
    results = []
    for i in top_indexes:
        results.append((float(scores[i]), int(i)))

    return results



def get_context(query, top_k=3):
    results = retrieve_similar_chunks(query, top_k=top_k)
  
    context_parts = []
    for score, idx in results:
        text = chunk_texts[idx]
        context_parts.append(text)

    print(context_parts)

    context = "\n\n".join(context_parts)
    return context

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13352.29it/s]


In [35]:
# print(get_context("What is the maximum loan amount for a personal loan?", top_k=3))

In [27]:

chain = GOOD_PROMPT | llm | StrOutputParser()

In [28]:
question = "how I can get the maximum loan amount for a personal loan?"

In [29]:
result = chain.invoke({
    "context": get_context(question, top_k=3),
    "question": question

})

['between 12 and 14 percent per annum, with a maximum loan amount of Rs 25 lakhs and tenure of 12 to\n60 months.\nFor applicants with a CIBIL score between 700 and 749, salaried borrowers are offered rates from 13 to\n16 percent per annum with a maximum loan of Rs 25 lakhs. Self-employed applicants in this bracket are', "Finance can lend. These limits are set by the Reserve Bank of India and cannot be exceeded under any\ncircumstances, regardless of the borrower's profile or the property type.\nFor loans up to Rs 30 lakhs, the maximum LTV is 90 percent of the property's value. For loans between", 'the last two years, a 12-month bank statement, and statements of all existing loans.\nFor amounts between Rs 25 lakhs and Rs 40 lakhs, all of the above are required along with proof of\ninvestments such as Fixed Deposits, Mutual Funds, or stock holdings, and property documents if\navailable.']


In [30]:
print(result)

To get the maximum loan amount of Rs 25 lakhs for a personal loan, you should have a CIBIL score of 700 or above. The interest rates for salaried borrowers in this bracket range from 13 to 16 percent per annum. Additionally, ensure you meet the required documentation criteria based on your loan amount.
